DATA FETCHING AND PREPROCESSING

In [ ]:
import os
import torch
import torchvision
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# ── Paths ──────────────────────────────────────────────────────────────────
# Switch this depending on environment
KAGGLE = False

if KAGGLE:
    DATA_ROOT = "/kaggle/input/ff-c23"
else:
    DATA_ROOT = r"C:\\Users\\fmatt\\OneDrive\\Desktop\\AI_lab\\deepfake-detection\\data"

# ── Device ─────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Data root: {DATA_ROOT}")

Using device: cpu
Data root: C:\\Users\\fmatt\\OneDrive\\Desktop\\AI_lab\\deepfake-detection\\data


In [6]:
import numpy as np
import PIL
from itertools import product

# define data paths
splits = ["train", "test", "val"]
classes= ["real", "fake"]
paths = [os.path.join(DATA_ROOT, split, cls) for split, cls in product(splits, classes)]

# filling folders with 5 dummy images
for path in paths:
    os.makedirs(path, exist_ok=True)
    for i in range(5):
        np_arr = np.random.randint(0,255, (224,224,3), dtype=np.uint8)
        img = PIL.Image.fromarray(np_arr)
        img.save(os.path.join(path, f"dummy_{i}.jpg"))

In [ ]:
# transformations for train/val/test

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transform  = transforms.Compose([transforms.RandomResizedCrop(224),
                                      transforms.RandomHorizontalFlip(),
                                      transforms.ToTensor(),
                                      transforms.Normalize(mean, std)])

val_transform = transforms.Compose([transforms.Resize(256),
                                   transforms.CenterCrop(224),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean, std)])

test_transform = transforms.Compose([transforms.Resize(256),
                                   transforms.CenterCrop(224),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean, std)])

In [ ]:
# data paths
train_path = os.path.join(DATA_ROOT, "train")
val_path = os.path.join(DATA_ROOT, "val")
test_path = os.path.join(DATA_ROOT, "test")

# datasets creation
train_dataset = ImageFolder(train_path, transform=train_transform)
val_dataset = ImageFolder(val_path, transform=val_transform)
test_dataset = ImageFolder(test_path, transform=test_transform)

# dataloader creation (train is shuffled)
train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=False)


In [ ]:
# Test to check everything works
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")

Train: 10 | Val: 10 | Test: 10
Classes: ['fake', 'real']


One important thing to note: ImageFolder assigned fake=0 and real=1 alphabetically. Keep this in mind later when interpreting model outputs.

In [ ]:
# checking size coherence
X, y = next(iter(train_dataloader))
X.shape, y.shape

(torch.Size([10, 3, 224, 224]), torch.Size([10]))

MODEL DEFINITION

In [ ]:
# Model definition
from torchvision.models import resnet50, ResNet50_Weights

"""
This downloads the ResNet-50 weights pretrained on ImageNet. T<
he weights are cached locally after the first download so you only download once.
"""
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\fmatt/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:10<00:00, 9.36MB/s]


In [13]:
"""
Replace fc of resnet50 of size (2048,1000) with a
fc of size (2048,2) since we are performing binary classification
"""
model.fc = nn.Linear(2048,2)
nn.init.xavier_uniform_(model.fc.weight)
model = model.to(device)

In [14]:
# define optimizer
params_1x = [param for name, param in model.named_parameters()
                              if not name.startswith("fc")]
optimizer = torch.optim.Adam([{"params": params_1x, "lr": 1e-4 },
                             {"params": model.fc.parameters(), "lr": 1e-3 }])

In [15]:
# mean reduction to average the loss across the batch (single GPU training)
loss_fn = nn.CrossEntropyLoss(reduction="mean") 


TRAINING LOOP

In [ ]:
num_epochs = 3
# training
for i in range(num_epochs):
    model.train() # set model in trainig mode ot activate specifi layers (batchnorm, dropout)
    train_loss = 0
    train_acc = 0
    for batch in train_dataloader:
        optimizer.zero_grad() # zeroth the gradients
        X,y = batch 
        X,y = X.to(device), y.to(device)
        y_hat = model(X) # forward pass
        batch_loss = loss_fn(y_hat, y) 
        preds = torch.max(y_hat, dim=1).indices # we want indices because y is made up of either 0 or 1
        batch_accuracy = torch.sum(y==preds)/y.shape[0]
        batch_loss.backward()  # backward pass
        optimizer.step() # params update
        train_loss += batch_loss
        train_acc += batch_accuracy
    train_loss = train_loss/len(train_dataloader)
    train_acc = train_acc/len(train_dataloader)
    print(f"train_loss epoch_{i}: {train_loss}")
    print(f"train_accuracy epoch_{i}: {train_acc}")
    
    # validation
    model.eval()
    val_loss = 0
    val_acc = 0
    for batch in val_dataloader:
        with torch.no_grad():
            X,y = batch 
            X,y = X.to(device), y.to(device)
            y_hat = model(X) # forward pass
            batch_loss = loss_fn(y_hat, y) 
            preds = torch.max(y_hat, dim=1).indices # we want indices because y is made up of either 0 or 1
            batch_accuracy = torch.sum(y==preds)/y.shape[0]
            val_loss += batch_loss
            val_acc += batch_accuracy
    val_loss = val_loss/len(val_dataloader)
    val_acc = val_acc/len(val_dataloader)
    print(f"val_loss epoch_{i}: {val_loss}")
    print(f"val_accuracy epoch_{i}: {val_acc}")

            